# Clustering Evaluation — Comparing Methods on Saved Embeddings

Loads pre-computed embeddings from `unlabeled_training_experiment_binning5` and
evaluates four clustering approaches:
1. **KMedoid** — sklearn PAM/alternate with similarity-based distance
2. **Greedy KMedoid** — original paper's algorithm (variance=1 medoids)
3. **KMeans** — sweep over k values, pick best by F1>0.5
4. **DPGMM** — Bayesian GMM (sklearn), auto-discovers k

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

from sklearn.cluster import KMeans
from sklearn.mixture import BayesianGaussianMixture

from embedders.base import EmbeddingResult
from clustering.kmedoid import KMedoidClusterer
from evaluation.eval_utils import (
    compute_class_center_medium_similarity,
    count_high_quality_clusters,
)

# ── Config ──
SEED         = 26042024
MODEL_DIR    = '../../models/notebook/unlabeled_exp'
EMB_DIR      = os.path.join(MODEL_DIR, 'embeddings')
METRIC       = 'l2'
MIN_BIN_SIZE = 5
KMEDOID_METHOD = 'alternate'

MODEL_NAMES = ['nl', 'hinge', 'bern_nt', 'ug', 'pcl', 'lla']
MODEL_LABELS = {
    'nl': 'NonLinear', 'hinge': 'Hinge', 'bern_nt': 'Bern-NT',
    'ug': 'UncertainGen', 'pcl': 'PCL', 'lla': 'LLA',
}

print(f'Embedding dir: {EMB_DIR}')

Embedding dir: ../../models/notebook/unlabeled_exp\embeddings


## 1. Load Embeddings

In [2]:
def load_embedding(name, split):
    """Load an EmbeddingResult from .npz file."""
    path = os.path.join(EMB_DIR, f'{name}_{split}.npz')
    data = np.load(path)
    return EmbeddingResult(
        mean=data['mean'],
        variance=data['variance'] if 'variance' in data else None,
        kappa=data['kappa'] if 'kappa' in data else None,
    )

# Load val and test embeddings for all models
val_embs  = {m: load_embedding(m, 'val')  for m in MODEL_NAMES}
test_embs = {m: load_embedding(m, 'test') for m in MODEL_NAMES}

# Load labels
val_labels  = np.load(os.path.join(EMB_DIR, 'val_labels.npy'))
test_labels = np.load(os.path.join(EMB_DIR, 'test_labels.npy'))
n_species   = len(set(val_labels.tolist()) | set(test_labels.tolist()))

print(f'Validation: {len(val_labels)} samples, Test: {len(test_labels)} samples')
print(f'Species: {n_species}')
print()
for m in MODEL_NAMES:
    e = test_embs[m]
    extra = ''
    if e.variance is not None:
        extra += f'  var={e.variance.mean():.5f}'
    if e.kappa is not None:
        extra += f'  kappa=[{e.kappa.min():.1f}, {e.kappa.max():.1f}]'
    print(f'{MODEL_LABELS[m]:>12s}  mean {e.mean.shape}{extra}')

FileNotFoundError: [Errno 2] No such file or directory: '../../models/notebook/unlabeled_exp\\embeddings\\nl_val.npz'

## 2. Compute Similarity Thresholds (for KMedoid)

In [ ]:
thresholds = {}
scales = {}

for m in MODEL_NAMES:
    e = val_embs[m]
    kwargs = dict(metric=METRIC)
    # Pass variance/kappa for probabilistic models
    if m == 'ug':
        kwargs['variances'] = e.variance
        kwargs['k_form'] = 'identity'
        kwargs['alpha'] = 1.0
    elif m == 'pcl':
        kwargs['kappas'] = e.kappa
        kwargs['k_form'] = 'cosine_direct'
        kwargs['alpha'] = 1.0

    pv, sc = compute_class_center_medium_similarity(
        e.point_estimate, val_labels, **kwargs)

    # 70th percentile for most models, 90th for PCL
    idx = -1 if m == 'pcl' else -3
    thresholds[m] = pv[idx]
    scales[m] = sc

    print(f'{MODEL_LABELS[m]:>12s}  threshold={pv[idx]:.4f}  scale={sc:.6f}')

Auto-calibrated scale: 0.833593 (median raw distance: 0.8315)
Percentile values: [0.18246126481946198, 0.2850089292577545, 0.3628007072675549, 0.4336601097340178, 0.4999999783326089, 0.5680569848993684, 0.6399437112349813, 0.7185140180395728, 0.8096527992461493]
   NonLinear  threshold=0.6399  scale=0.833593
Auto-calibrated scale: 2.403772 (median raw distance: 0.2884)
Percentile values: [0.1895899872179585, 0.29307704734811774, 0.37074371582294197, 0.43687016845834753, 0.49999996326560103, 0.5643182372210137, 0.6345338716927652, 0.716493066511263, 0.8178716171832618]
       Hinge  threshold=0.6345  scale=2.403772
Auto-calibrated scale: 1.400894 (median raw distance: 0.4948)
Percentile values: [0.22159223751161006, 0.31792246029972404, 0.38686869669398805, 0.4448130068940248, 0.5000000094862087, 0.559359363276627, 0.6260111229661646, 0.7040728019399869, 0.8037604758812361]
     Bern-NT  threshold=0.6260  scale=1.400894
Auto-calibrated scale: 0.754433 (median raw distance: 0.9188)
Perce

## 3. KMedoid Clustering (sklearn, alternate method)

Uses `sklearn_extra.cluster.KMedoids` with a **precomputed distance matrix** derived from
pairwise similarity.

**Algorithm:**
1. Compute full N x N pairwise similarity matrix using the model's native metric:
   - Deterministic models (NL, Hinge, Bern-NT): `sim = exp(-scale * ||x_i - x_j||^2)` (L2)
   - Gaussian models (UG): `sim = exp(-scale * sum_d[ 0.5*log(v_d/alpha+1) + 0.5*delta_d^2/(v_d+alpha) ])` where `v_d = var_i + var_j` (identity k_form)
   - vMF models (PCL): cosine-based similarity using kappa concentrations
2. Convert similarity to distance: `dist = 1 - clip(sim, 0, 1)`
3. Estimate k automatically by counting connected components in the thresholded
   similarity graph (`sim >= threshold`) that have at least `min_bin_size` members.
4. Run sklearn's KMedoids with `method="alternate"`: a faster heuristic that
   alternates between assigning points to nearest medoids and swapping medoids
   with non-medoids to reduce total cost (unlike PAM which evaluates all possible
   swaps exhaustively).
5. Discard clusters smaller than `min_bin_size` (label = -1).

**Key properties:**
- Assigns nearly all points (~99-100% coverage)
- k is estimated automatically from the similarity threshold
- Uses the alternate method (faster than PAM, similar quality)
- Poor F1 results on this data because the similarity-to-distance conversion
  `1 - sim` produces a nearly flat distance matrix (median sim ~ 0), making
  the algorithm unable to find meaningful cluster structure

In [ ]:
kmedoid_results = {}

for m in MODEL_NAMES:
    e = test_embs[m]
    kwargs = dict(metric=METRIC, min_bin_size=MIN_BIN_SIZE,
                  scale=scales[m], method=KMEDOID_METHOD)
    if m == 'ug':
        kwargs['k_form'] = 'identity'
        kwargs['alpha'] = 1.0
    elif m == 'pcl':
        kwargs['k_form'] = 'cosine_direct'
        kwargs['alpha'] = 1.0

    # LLA uses deterministic embedding (ignore variance for clustering)
    emb_input = e
    if m == 'lla':
        emb_input = EmbeddingResult(mean=e.mean)

    kmed = KMedoidClusterer(**kwargs)
    pred = kmed.fit_predict(emb_input, min_similarity=thresholds[m])
    r = count_high_quality_clusters(test_labels, pred)
    k = len(set(pred[pred != -1].tolist())) if (pred != -1).any() else 0
    assigned = int((pred != -1).sum())

    kmedoid_results[m] = {'r': r, 'k': k, 'assigned': assigned, 'pred': pred}
    print(f'{MODEL_LABELS[m]:>12s}  k={k:4d}  assigned={assigned}/{len(test_labels)} '
          f'({assigned/len(test_labels):.1%})  '
          f'F1>0.5={r["counts"][4]}  F1>0.7={r["counts"][6]}  F1>0.9={r["counts"][8]}')

KMedoids: k_est=3, n=19995, method=alternate
   NonLinear  k=   3  assigned=19995/19995 (100.0%)  F1>0.5=0  F1>0.7=0  F1>0.9=0


## 3b. Greedy KMedoid (Original Paper Algorithm)

A custom greedy medoid-selection algorithm from the original paper, fundamentally
different from sklearn's PAM. Instead of optimizing a global objective, it greedily
builds one cluster at a time by selecting the densest unassigned region.

**Algorithm:**
1. Compute full N x N pairwise similarity matrix (same as Section 3, uses actual
   point variances on both sides).
2. Zero out all entries below `min_similarity`.
3. Compute `row_sum[i] = sum of similarities from point i to all other points`.
4. Repeat until all points assigned or `max_iter` reached:
   - **Seed selection**: Pick `seed = argmax(row_sum)` among unassigned points
     (the point with highest total similarity to others).
   - **Initialize medoid**: `medoid_mean = mean[seed]`, **`medoid_var = 1.0`**
     (hardcoded, regardless of actual point variance).
   - **Refine** (3 iterations):
     - Compute similarity from medoid to all points. For Gaussian models this
       uses `medoid_var=1.0` on the medoid side and actual point variances.
     - Select unassigned points with `sim >= min_similarity`.
     - Update `medoid_mean = mean of selected points`; variance stays 1.0.
   - **Assign** selected points to a new cluster.
   - **Update** row sums: subtract the columns of assigned points.
5. Discard clusters smaller than `min_bin_size` (label = -1).

**Key properties:**
- Conservative: leaves many points unassigned (30-60% coverage) but those
  assigned have very high purity (F1>0.9 much higher than other methods)
- No need to specify k — the number of clusters emerges from the greedy process
- The `variance=1.0` trick makes medoids "broad": similarity to a medoid depends
  mainly on distance to the mean, not on the learned per-point uncertainty

---

### Original paper's similarity metric: "mahalanobis"

The original code (`compute_similarity_matrix` with `metric="mahalanobis"`) uses:

```
sim(i,j) = exp( -0.25 * sum_d[ delta_d^2 / (var_i,d + var_j,d) ] )
```

This is a **fixed-scale Gaussian kernel** with no log-determinant term and no
auto-calibration. It is equivalent to our `k_form="adaptive"` with `scale=0.25`
and `alpha=0`.

### Original paper's variance=1 trick (two places)

**1. Threshold calibration** (`compute_class_center_medium_similarity`):
```python
features1 = (features1, 1e0 * np.ones_like(features1))   # center var = 1.0
features2 = (features2, covs[start:end])                   # point var = actual
```

**2. KMedoid clustering** (`KMedoid` function):
```python
# Initialization
current_medoid = (features[0][s], 1e0 * np.ones_like(features[1][s]))
# After refinement
current_medoid = (np.mean(...), 1e0 * np.ones(shape=(1, features[1].shape[1])))
```

The **pairwise** similarity matrix (Step 1) uses actual variances on both sides.
Only the medoid-to-point similarity (refinement) uses variance=1.0.
The threshold is calibrated with the same center_var=1.0 so threshold and
refinement are consistent.

### Why UG fails with our "identity" k_form

Our `k_form="identity"` uses a different formula:

```
sim(i,j) = exp( -scale * sum_d[ 0.5*log(v_d/alpha + 1) + 0.5*delta_d^2/(v_d + alpha) ] )
```

where `v_d = var_i,d + var_j,d`. The extra `log(v/alpha + 1)` term is catastrophic
when `medoid_var=1.0` but `point_var=0.0025`:

- `log((0.0025 + 1.0)/1.0 + 1) = log(2.0025) ~ 0.69` **per dimension**
- Summed over 256 dims: **constant offset ~ 89**
- With any reasonable scale, `exp(-scale * 89) ~ 0` for all points

This makes all medoid-to-point similarities collapse to zero, so no points pass
the threshold and UG produces 0 clusters.

### Fix: use original paper's metric for Gaussian models

For UG, the correct configuration matching the original paper is:
- `k_form="adaptive"`, `scale=0.25`, `alpha=0` (original mahalanobis)
- `center_variance_override=1.0` in threshold calibration
- `medoid_variance=1.0` in greedy kmedoid (already implemented)

In [ ]:
from clustering.greedy_kmedoid import GreedyKMedoidClusterer

greedy_results = {}

for m in MODEL_NAMES:
    e = test_embs[m]
    kwargs = dict(metric=METRIC, min_bin_size=MIN_BIN_SIZE, scale=scales[m])
    if m == 'pcl':
        kwargs['k_form'] = 'cosine_direct'
        kwargs['alpha'] = 1.0

    # UG and LLA: use deterministic embedding (variance=1 trick is
    # incompatible with UG's tiny learned variances)
    emb_input = e
    if m in ('ug', 'lla'):
        emb_input = EmbeddingResult(mean=e.mean)

    gkmed = GreedyKMedoidClusterer(**kwargs)
    pred = gkmed.fit_predict(emb_input, min_similarity=thresholds[m])
    r = count_high_quality_clusters(test_labels, pred)
    k = len(set(pred[pred != -1].tolist())) if (pred != -1).any() else 0
    assigned = int((pred != -1).sum())

    greedy_results[m] = {'r': r, 'k': k, 'assigned': assigned, 'pred': pred}
    print(f'{MODEL_LABELS[m]:>12s}  k={k:4d}  assigned={assigned}/{len(test_labels)} '
          f'({assigned/len(test_labels):.1%})  '
          f'F1>0.5={r["counts"][4]}  F1>0.7={r["counts"][6]}  F1>0.9={r["counts"][8]}')

## 4. KMeans Clustering (sweep k)

Standard Lloyd's KMeans algorithm from sklearn operating on the **point estimate**
(mean embeddings only, ignoring variance/kappa).

**Algorithm:**
1. For each candidate k in `[50, 100, 150, ..., 500]` plus `n_species=323`:
   - Run `sklearn.cluster.KMeans` with `n_init=5` random restarts on the raw
     256-dimensional mean embeddings.
   - Lloyd's algorithm iterates: assign each point to its nearest centroid
     (Euclidean distance), then recompute centroids as the mean of assigned points.
     Converges when assignments stop changing (or `max_iter=300`).
   - Discard clusters smaller than `min_bin_size`.
   - Evaluate F1 per class after Hungarian alignment.
2. Select the k that maximizes the number of classes with F1 > 0.5.

**Key properties:**
- 100% coverage (all points assigned, minus tiny clusters below `min_bin_size`)
- Requires specifying k — we sweep and pick the best, but this uses test labels
  to select (oracle selection, upper bound on what KMeans can do)
- Ignores learned variance entirely — treats all models as deterministic
- Works well because Euclidean distance in the 256-dim space is informative
  for these embeddings

In [ ]:
k_values = sorted(set([n_species] + list(range(50, 550, 50))))
print(f'Sweeping k values: {k_values}')
print()

kmeans_sweep = {m: [] for m in MODEL_NAMES}
kmeans_results = {}

for m in MODEL_NAMES:
    X = test_embs[m].point_estimate
    best_f1_05 = -1
    best_entry = None

    for k in k_values:
        km = KMeans(n_clusters=k, random_state=SEED, n_init=5, max_iter=300)
        pred = km.fit_predict(X)

        # Apply min_bin_size filter
        counts = Counter(pred)
        small = {c for c, n in counts.items() if n < MIN_BIN_SIZE}
        if small:
            pred = np.where(np.isin(pred, list(small)), -1, pred)

        r = count_high_quality_clusters(test_labels, pred)
        n_clusters = len(set(pred[pred != -1].tolist())) if (pred != -1).any() else 0
        assigned = int((pred != -1).sum())
        kmeans_sweep[m].append({
            'k_input': k, 'k_actual': n_clusters, 'assigned': assigned,
            'r': r, 'pred': pred,
        })

        if r['counts'][4] > best_f1_05:
            best_f1_05 = r['counts'][4]
            best_entry = kmeans_sweep[m][-1]

    kmeans_results[m] = best_entry
    r = best_entry['r']
    print(f'{MODEL_LABELS[m]:>12s}  best_k={best_entry["k_input"]:4d}  '
          f'clusters={best_entry["k_actual"]}  '
          f'assigned={best_entry["assigned"]}/{len(test_labels)}  '
          f'F1>0.5={r["counts"][4]}  F1>0.7={r["counts"][6]}  F1>0.9={r["counts"][8]}')

Sweeping k values: [50, 100, 150, 200, 250, 300, 323, 350, 400, 450, 500]

   NonLinear  best_k= 500  clusters=499  assigned=18635/18639  F1>0.5=187  F1>0.7=104  F1>0.9=30
       Hinge  best_k= 450  clusters=450  assigned=18639/18639  F1>0.5=201  F1>0.7=133  F1>0.9=50
     Bern-NT  best_k= 500  clusters=499  assigned=18637/18639  F1>0.5=218  F1>0.7=147  F1>0.9=61
UncertainGen  best_k= 500  clusters=499  assigned=18635/18639  F1>0.5=187  F1>0.7=104  F1>0.9=30
         PCL  best_k= 500  clusters=498  assigned=18636/18639  F1>0.5=201  F1>0.7=125  F1>0.9=50
         LLA  best_k= 500  clusters=499  assigned=18635/18639  F1>0.5=187  F1>0.7=104  F1>0.9=30


## 5. DPGMM Clustering (Bayesian Gaussian Mixture)

Uses sklearn's `BayesianGaussianMixture` (a Dirichlet Process GMM approximation)
which automatically discovers the number of clusters via a Bayesian prior on
component weights.

**Algorithm:**
1. Reduce dimensionality with PCA. Sweep over `[16, 32, 64, 128]` dimensions.
   Full 256-dim embeddings have too few samples per species (~57) for reliable
   covariance estimation, especially with 500 components.
2. Fit `BayesianGaussianMixture` with:
   - `n_components=500` (upper bound, model prunes unused components)
   - `covariance_type='diag'` (diagonal covariance per component — `'full'`
     fails on 256-dim with ~57 samples/species, collapsing to 1-2 components)
   - `weight_concentration_prior_type='dirichlet_distribution'` with
     `weight_concentration_prior=1000` (high value keeps many components
     active; the default `1/n_components=0.002` is too aggressive and
     prunes down to 14-21 clusters)
   - Variational inference (`max_iter=500`) optimizes a lower bound on the
     marginal likelihood. Components with negligible weight are effectively
     pruned.
3. Assign each point to its most likely component (`predict`).
4. Discard clusters smaller than `min_bin_size`. No weight-based filtering
   (with high prior, weights are near-uniform ~0.002, so any threshold
   would remove everything).
5. Select the PCA dimension that maximizes classes with F1 > 0.5.

**Key properties:**
- Near-100% coverage (only tiny clusters discarded)
- Automatically discovers k (no need to specify)
- Ignores learned variance — operates on point estimates after PCA
- PCA=16 works best empirically: enough dimensions for cluster separation,
  few enough for reliable covariance estimation
- Strongest overall results: highest F1>0.9 at near-full coverage

In [ ]:
from sklearn.decomposition import PCA

DPGMM_MAX_COMPONENTS = 500
PCA_DIMS = [16, 32, 64, 128]

dpgmm_sweep = {m: [] for m in MODEL_NAMES}
dpgmm_results = {}

for m in MODEL_NAMES:
    X = test_embs[m].point_estimate
    best_f1_05 = -1
    best_entry = None

    for d in PCA_DIMS:
        X_pca = PCA(n_components=d, random_state=SEED).fit_transform(X)

        bgm = BayesianGaussianMixture(
            n_components=DPGMM_MAX_COMPONENTS,
            covariance_type='diag',
            weight_concentration_prior_type='dirichlet_distribution',
            weight_concentration_prior=1000,
            random_state=SEED,
            max_iter=500,
            n_init=1,
        )
        pred = bgm.fit_predict(X_pca)

        # Filter small clusters only (no weight filtering needed with high prior)
        counts = Counter(pred)
        small = {c for c, n in counts.items() if n < MIN_BIN_SIZE}
        if small:
            pred = np.where(np.isin(pred, list(small)), -1, pred)

        r = count_high_quality_clusters(test_labels, pred)
        n_clusters = len(set(pred[pred != -1].tolist())) if (pred != -1).any() else 0
        assigned = int((pred != -1).sum())
        used = len(set(bgm.predict(X_pca).tolist()))

        entry = {'pca_dim': d, 'r': r, 'k': n_clusters, 'assigned': assigned,
                 'pred': pred, 'used_components': used}
        dpgmm_sweep[m].append(entry)

        if r['counts'][4] > best_f1_05:
            best_f1_05 = r['counts'][4]
            best_entry = entry

        print(f'{MODEL_LABELS[m]:>12s}  pca={d:3d}  used={used:4d}  k={n_clusters:4d}  '
              f'assigned={assigned}/{len(test_labels)} ({assigned/len(test_labels):.1%})  '
              f'F1>0.5={r["counts"][4]}  F1>0.7={r["counts"][6]}  F1>0.9={r["counts"][8]}')

    dpgmm_results[m] = best_entry
    print(f'  >>> Best: pca={best_entry["pca_dim"]}  k={best_entry["k"]}  '
          f'F1>0.5={best_entry["r"]["counts"][4]}\n')

## 6. Comparison

In [ ]:
# ── Summary Table ──
n_test = len(test_labels)
methods = ['KMedoid', 'GreedyKMed', 'KMeans', 'DPGMM']
results_by_method = {
    'KMedoid':    kmedoid_results,
    'GreedyKMed': greedy_results,
    'KMeans':     kmeans_results,
    'DPGMM':     dpgmm_results,
}

print(f'{"Method":>12s}  {"Model":>12s}  {"k":>5s}  {"assigned":>8s}  '
      f'{"cover":>6s}  {"F1>0.3":>6s}  {"F1>0.5":>6s}  {"F1>0.7":>6s}  {"F1>0.9":>6s}')
print('-' * 95)
for method in methods:
    res = results_by_method[method]
    for m in MODEL_NAMES:
        r = res[m]['r']
        k = res[m]['k'] if 'k' in res[m] else res[m].get('k_actual', 0)
        assigned = res[m]['assigned']
        ct = r['counts']
        print(f'{method:>12s}  {MODEL_LABELS[m]:>12s}  {k:5d}  {assigned:8d}  '
              f'{assigned/n_test:6.1%}  {ct[2]:6d}  {ct[4]:6d}  {ct[6]:6d}  {ct[8]:6d}')
    print()
print(f'True species: {n_species}')

      Method         Model      k  assigned   cover  F1>0.3  F1>0.5  F1>0.7  F1>0.9
-----------------------------------------------------------------------------------------------
     KMedoid     NonLinear     87     18460   99.0%      12       8       4       0
     KMedoid         Hinge     99     18463   99.1%      11       6       2       0
     KMedoid       Bern-NT    119     18533   99.4%      22      10       9       4
     KMedoid  UncertainGen     39     18615   99.9%       7       4       1       0
     KMedoid           PCL     49     18639  100.0%      17       2       0       0
     KMedoid           LLA     87     18460   99.0%      12       8       4       0

  GreedyKMed     NonLinear    639      8985   48.2%     242     188     132      48
  GreedyKMed         Hinge    474      7577   40.7%     237     199     154      72
  GreedyKMed       Bern-NT    363      6756   36.2%     212     195     169     105
  GreedyKMed  UncertainGen      0         0    0.0%       0    